# RAPIDS GPU-Accelerated Data Science Demo

End-to-end demonstration of GPU acceleration using NVIDIA RAPIDS for DataFrame operations, machine learning, text embeddings, and similarity search. Each section compares GPU vs CPU performance.

## Setup

In [ ]:
import time
import os
import numpy as np
import cupy as cp
import cudf
import pandas as pd

print("=" * 60)
print("HARDWARE")
print("=" * 60)
os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"\ncuDF version: {cudf.__version__}")
print(f"CuPy version: {cp.__version__}")

## 1. DataFrame Operations: cuDF vs pandas

Operations on 100M rows. cuDF keeps data in GPU memory and processes all rows in parallel.

In [ ]:
num_rows = 100_000_000
rng = np.random.default_rng(seed=42)

print(f"Creating DataFrame with {num_rows:,} rows...")
pdf = pd.DataFrame({
    "numbers": rng.standard_normal(num_rows).astype(np.float32),
    "category": rng.integers(0, 1000, num_rows).astype(np.int32),
})
gdf = cudf.DataFrame(pdf)
print(f"Done. Size in memory: ~{pdf.memory_usage(deep=True).sum()/1e9:.1f} GB")

# Warmup GPU (first operation includes initialization overhead)
_ = gdf["category"].sum()
cp.cuda.Stream.null.synchronize()

results = {}

# value_counts
start = time.perf_counter()
_ = pdf["category"].value_counts()
cpu_time = time.perf_counter() - start

start = time.perf_counter()
_ = gdf["category"].value_counts()
cp.cuda.Stream.null.synchronize()
gpu_time = time.perf_counter() - start
results["value_counts"] = (cpu_time, gpu_time)
print(f"value_counts:  CPU {cpu_time:.3f}s  GPU {gpu_time:.3f}s  Speedup: {cpu_time/gpu_time:.0f}x")

# groupby mean
start = time.perf_counter()
_ = pdf.groupby("category")["numbers"].mean()
cpu_time = time.perf_counter() - start

start = time.perf_counter()
_ = gdf.groupby("category")["numbers"].mean()
cp.cuda.Stream.null.synchronize()
gpu_time = time.perf_counter() - start
results["groupby_mean"] = (cpu_time, gpu_time)
print(f"groupby mean:  CPU {cpu_time:.3f}s  GPU {gpu_time:.3f}s  Speedup: {cpu_time/gpu_time:.0f}x")

# sort
start = time.perf_counter()
_ = pdf.sort_values("numbers")
cpu_time = time.perf_counter() - start

start = time.perf_counter()
_ = gdf.sort_values("numbers")
cp.cuda.Stream.null.synchronize()
gpu_time = time.perf_counter() - start
results["sort"] = (cpu_time, gpu_time)
print(f"sort:          CPU {cpu_time:.3f}s  GPU {gpu_time:.3f}s  Speedup: {cpu_time/gpu_time:.0f}x")

## 2. String Operations

String processing is where GPU shows the most extreme speedups — each character operation on each row runs in parallel.

In [ ]:
import gc
del pdf, gdf
gc.collect()

num_rows = 50_000_000
print(f"Creating string Series with {num_rows:,} rows...")

pd_series = pd.Series(
    rng.choice(
        ["hello world", "RAPIDS is fast", "gpu acceleration", "data science", "machine learning"],
        size=num_rows,
    )
)
gd_series = cudf.Series(pd_series)

# String contains
start = time.perf_counter()
_ = pd_series.str.contains("fast")
cpu_time = time.perf_counter() - start

start = time.perf_counter()
_ = gd_series.str.contains("fast")
cp.cuda.Stream.null.synchronize()
gpu_time = time.perf_counter() - start
print(f"str.contains:  CPU {cpu_time:.3f}s  GPU {gpu_time:.3f}s  Speedup: {cpu_time/gpu_time:.0f}x")

# String upper
start = time.perf_counter()
_ = pd_series.str.upper()
cpu_time = time.perf_counter() - start

start = time.perf_counter()
_ = gd_series.str.upper()
cp.cuda.Stream.null.synchronize()
gpu_time = time.perf_counter() - start
print(f"str.upper:     CPU {cpu_time:.3f}s  GPU {gpu_time:.3f}s  Speedup: {cpu_time/gpu_time:.0f}x")

## 3. Machine Learning: KMeans Clustering

2M points, 500 clusters — GPU computes all distances in parallel.

In [ ]:
del pd_series, gd_series
gc.collect()

from cuml.cluster import KMeans as cuKMeans
from sklearn.cluster import KMeans as skKMeans

n_samples = 2_000_000
n_features = 50
n_clusters = 500

X_cpu = rng.standard_normal((n_samples, n_features)).astype(np.float32)
X_gpu = cp.asarray(X_cpu)

print(f"KMeans: {n_samples:,} samples, {n_features} features, {n_clusters} clusters")

start = time.perf_counter()
skKMeans(n_clusters=n_clusters, max_iter=10, n_init=1, random_state=42).fit(X_cpu)
cpu_time = time.perf_counter() - start

start = time.perf_counter()
cuKMeans(n_clusters=n_clusters, max_iter=10, n_init=1, random_state=42).fit(X_gpu)
cp.cuda.Stream.null.synchronize()
gpu_time = time.perf_counter() - start

print(f"KMeans:        CPU {cpu_time:.3f}s  GPU {gpu_time:.3f}s  Speedup: {cpu_time/gpu_time:.0f}x")

## 4. Random Forest Classification

100 trees built in parallel on GPU.

In [ ]:
del X_cpu, X_gpu
gc.collect()

from cuml.ensemble import RandomForestClassifier as cuRF
from sklearn.ensemble import RandomForestClassifier as skRF

n_samples = 500_000
n_features = 50
X_cpu = rng.standard_normal((n_samples, n_features)).astype(np.float32)
y_cpu = rng.integers(0, 5, n_samples).astype(np.int32)
X_gpu = cudf.DataFrame(X_cpu)
y_gpu = cudf.Series(y_cpu)

print(f"Random Forest: {n_samples:,} samples, {n_features} features, 100 trees")

start = time.perf_counter()
skRF(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1).fit(X_cpu, y_cpu)
cpu_time = time.perf_counter() - start

start = time.perf_counter()
cuRF(n_estimators=100, max_depth=12, random_state=42).fit(X_gpu, y_gpu)
cp.cuda.Stream.null.synchronize()
gpu_time = time.perf_counter() - start

print(f"Random Forest: CPU {cpu_time:.3f}s  GPU {gpu_time:.3f}s  Speedup: {cpu_time/gpu_time:.0f}x")

## 5. Text Embedding & KNN Similarity Search

Generate text embeddings on GPU using SentenceTransformers, then find nearest neighbors using cuML.

In [ ]:
del X_cpu, X_gpu, y_cpu, y_gpu
gc.collect()

import warnings, logging
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from sentence_transformers import SentenceTransformer
from cuml.neighbors import NearestNeighbors as cuNearestNeighbors

# Generate synthetic documents
import random
random.seed(42)
categories = ["electronics", "food", "clothing", "books", "toys"]
adjectives_good = ["amazing", "excellent", "fantastic", "great", "wonderful"]
adjectives_bad = ["terrible", "awful", "poor", "disappointing", "broken"]
templates = [
    "This {cat} product is {adj}. Highly recommend!",
    "I bought this {cat} item and it was {adj}.",
    "The {cat} quality is {adj}, will buy again.",
    "{adj} {cat} purchase, exactly what I needed.",
]

documents = []
for _ in range(10000):
    cat = random.choice(categories)
    adj = random.choice(adjectives_good + adjectives_bad)
    tmpl = random.choice(templates)
    documents.append(tmpl.format(cat=cat, adj=adj))

print(f"Generated {len(documents):,} documents")
print(f"Sample: {documents[0]}")

In [ ]:
# Generate embeddings on GPU
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

start = time.perf_counter()
embeddings = model.encode(documents, batch_size=256, show_progress_bar=False, convert_to_numpy=True)
gpu_time = time.perf_counter() - start
print(f"Embedding {len(documents):,} docs on GPU: {gpu_time:.2f}s ({len(documents)/gpu_time:.0f} docs/sec)")
print(f"Embedding shape: {embeddings.shape}")

In [ ]:
# KNN search on GPU
queries = [
    "delicious food with amazing taste",
    "terrible electronics that broke quickly",
    "comfortable sports clothing",
]

query_embeddings = model.encode(queries, convert_to_numpy=True)
k = 5

# GPU KNN
embeddings_gpu = cp.asarray(embeddings)
query_gpu = cp.asarray(query_embeddings)

start = time.perf_counter()
knn = cuNearestNeighbors(n_neighbors=k, metric='cosine')
knn.fit(embeddings_gpu)
distances, indices = knn.kneighbors(query_gpu)
gpu_time = time.perf_counter() - start
print(f"GPU KNN search ({len(queries)} queries, {len(documents):,} corpus): {gpu_time:.4f}s")

# Show results
print("\n" + "=" * 60)
for i, query in enumerate(queries):
    print(f"\nQuery: \"{query}\"")
    print(f"  Top {k} matches:")
    for j in range(k):
        idx = int(indices[i][j])
        dist = float(distances[i][j])
        print(f"    {j+1}. [{dist:.3f}] {documents[idx]}")

## Summary

| Category | Operation | Expected Speedup |
|----------|-----------|------------------|
| **DataFrame** | value_counts, groupby, sort | 30–100x |
| **String** | contains, upper | 50–150x |
| **ML** | KMeans (2M points) | 20–50x |
| **ML** | Random Forest (500K rows) | 5–15x |
| **Embedding** | SentenceTransformer encode | 10–30x vs CPU |
| **KNN Search** | cuML NearestNeighbors | 50–200x |